# 00 - Shared evaluation protocol demo

Shows how every model in this project is scored (issue #8). Copy this pattern in your own notebook.

**Rules**
- Develop and tune on `load_dev()` (train + val). Never touch the test set until the final comparison (#18).
- Score with `evaluate(...)`: repeated stratified 5-fold CV x 3, seed 42, same metrics for everyone.
- Put scaling or other preprocessing **inside a Pipeline** so it is fit on training folds only.
- `evaluate(..., log=True)` appends a row to `results/model_results.csv`. Commit that file with your PR.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # so `src` is importable from notebooks/

import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

from src.common import load_dev, evaluate, RANDOM_STATE

## 1. Load the development data (6 raw sensors)

In [ ]:
X, y = load_dev()          # features='all' adds the 16 engineered columns (issue #14)
print(X.shape, 'share Needs Maintenance:', round(y.mean(), 3))
X.head()

## 2. Score two models

This demo uses `log=False` so it does not write to the shared results file. In your own notebook leave `log=True` (the default).

In [ ]:
results = [
    evaluate(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
             X, y, name='LogisticRegression', params={'C': 1.0}, log=False),
    evaluate(HistGradientBoostingClassifier(random_state=RANDOM_STATE),
             X, y, name='HistGradientBoosting', params={'defaults': True}, log=False),
]
cols = ['model', 'cv_auc_mean', 'cv_auc_std', 'cv_pr_auc_mean', 'cv_f1_mean', 'cv_recall_mean', 'fit_time_mean_s']
pd.DataFrame(results)[cols]

## 3. Reading the results
Differences between models here are small (AUC about 0.69). Compare the gap with the std across folds before calling one model better.

## 4. Test set
Do **not** load it here. For the final comparison only: `from src.common import load_test; X_test, y_test = load_test(confirm=True)`.